# Travel Reimbursement Approval Agent

A runnable, policy-grounded agentic prototype for reviewing the five sample claims in the assignment brief. The notebook is deliberately self-contained and uses deterministic policy tools for the financial decision, with an optional LLM narration hook that is disabled by default.

## README — setup and run

1. Open this notebook in Jupyter, JupyterLab, or VS Code.
2. Run all cells from top to bottom. No package installation or API key is required for the default path.
3. The last cell prints the required JSON array for all five claims.
4. Optional LLM narration: set `OPENAI_API_KEY` and change `ENABLE_LLM_NARRATION = True` in the configuration cell. The LLM only rewrites evidence into an explanation; the policy tools remain authoritative.

The notebook also renders a compact dashboard and saves `UI_SS_1.png` next to the notebook when Matplotlib is available.

## Design overview

The `TravelReimbursementAgent` uses a small planner to select policy tools, retrieves the relevant rule IDs, runs each tool, and synthesizes a structured result. This makes the workflow inspectable and reliable: the model-like synthesis layer cannot invent policy limits because the limits and decision gates are encoded in tool outputs.

Toolbox: `policy_lookup`, `receipt_completeness_check`, `limit_checker`, `approval_threshold_check`, `timeliness_checker`, and `output_validator`. Manual review is preferred whenever evidence is incomplete, an airfare exception exists, the submission is late, or the amount exceeds the agent's authority.

In [ ]:
import json
import os
from datetime import date
from copy import deepcopy

ENABLE_LLM_NARRATION = False  # Optional: requires OPENAI_API_KEY and an internet-enabled runtime.
TODAY = date(2026, 8, 29)
ALLOWED_DECISIONS = {'APPROVE', 'PARTIAL_APPROVE', 'REJECT', 'MANUAL_REVIEW'}
REQUIRED_FIELDS = {'claim_id', 'decision', 'approved_amount', 'deducted_amount', 'missing_docs', 'policy_refs', 'confidence', 'explanation', 'tools_used'}

POLICY = {
    'eligible_categories': {'airfare', 'lodging', 'meals', 'ground_transport', 'conference_fees'},
    'ineligible_categories': {'alcohol', 'minibar', 'spa', 'gym', 'personal_entertainment', 'personal_shopping', 'gifts', 'traffic_fines', 'penalties', 'late_fees', 'personal'},
    'meal_daily_cap': 75.0,
    'lodging_nightly_cap': 200.0,
    'ground_daily_cap': 50.0,
    'receipt_threshold': 25.0,
    'submission_window_days': 30,
    'high_value_threshold': 2000.0,
    'rules': {
        'airfare': ['POL-CAT-01', 'POL-AIR-01', 'POL-RCT-01'],
        'lodging': ['POL-CAT-01', 'POL-PD-02', 'POL-RCT-01'],
        'meals': ['POL-CAT-01', 'POL-PD-01', 'POL-RCT-01'],
        'ground_transport': ['POL-CAT-01', 'POL-PD-03', 'POL-RCT-01'],
        'conference_fees': ['POL-CAT-01', 'POL-RCT-01'],
        'ineligible': ['POL-CAT-02'],
    },
}


## Provided claims

These are the five claims from Appendix B. No additional claims are introduced.

In [ ]:
CLAIMS = [
    {
        'claim_id': 'CLM-001', 'employee': 'A. Rivera',
        'trip_start': '2026-06-10', 'trip_end': '2026-06-12', 'submitted': '2026-06-20',
        'items': [
            {'category': 'airfare', 'description': 'Round-trip economy airfare', 'amount': 420.00, 'receipt_attached': True, 'class': 'economy'},
            {'category': 'lodging', 'description': 'Hotel, 2 nights @ $180', 'amount': 360.00, 'receipt_attached': True, 'nights': 2},
            {'category': 'meals', 'description': 'Meals, 3 days @ ~$60/day', 'amount': 180.00, 'receipt_attached': True, 'days': 3},
            {'category': 'conference_fees', 'description': 'Conference registration', 'amount': 150.00, 'receipt_attached': True},
        ],
    },
    {
        'claim_id': 'CLM-002', 'employee': 'B. Osei',
        'trip_start': '2026-06-14', 'trip_end': '2026-06-15', 'submitted': '2026-06-25',
        'items': [
            {'category': 'spa', 'description': 'Hotel spa package', 'amount': 300.00, 'receipt_attached': True},
            {'category': 'minibar', 'description': 'In-room minibar', 'amount': 80.00, 'receipt_attached': True},
        ],
    },
    {
        'claim_id': 'CLM-003', 'employee': 'C. Nakamura',
        'trip_start': '2026-06-08', 'trip_end': '2026-06-10', 'submitted': '2026-06-22',
        'items': [
            {'category': 'airfare', 'description': 'Round-trip economy airfare', 'amount': 300.00, 'receipt_attached': True, 'class': 'economy'},
            {'category': 'lodging', 'description': 'Hotel, 2 nights @ $250', 'amount': 500.00, 'receipt_attached': True, 'nights': 2},
            {'category': 'meals', 'description': 'Meals, 2 days @ $70/day', 'amount': 140.00, 'receipt_attached': True, 'days': 2},
        ],
    },
    {
        'claim_id': 'CLM-004', 'employee': 'D. Fischer',
        'trip_start': '2026-06-16', 'trip_end': '2026-06-18', 'submitted': '2026-06-28',
        'items': [
            {'category': 'airfare', 'description': 'Business-class international airfare', 'amount': 2400.00, 'receipt_attached': True, 'class': 'business'},
            {'category': 'lodging', 'description': 'Hotel, 3 nights', 'amount': 600.00, 'receipt_attached': False, 'nights': 3},
        ],
    },
    {
        'claim_id': 'CLM-005', 'employee': 'E. Haddad',
        'trip_start': '2026-06-11', 'trip_end': '2026-06-11', 'submitted': '2026-06-24',
        'items': [
            {'category': 'meals', 'description': 'Client dinner for 4 (business development)', 'amount': 220.00, 'receipt_attached': False, 'days': 1},
        ],
    },
]

assert len(CLAIMS) == 5
assert [c['claim_id'] for c in CLAIMS] == ['CLM-001', 'CLM-002', 'CLM-003', 'CLM-004', 'CLM-005']
CLAIMS_JSON = json.dumps(CLAIMS)
CLAIMS = json.loads(CLAIMS_JSON)
assert isinstance(CLAIMS, list) and isinstance(CLAIMS[0], dict)

## Agent tools and orchestration

Each tool returns structured evidence and the rule IDs that support it. The agent planner decides which tools are needed from the claim contents; the synthesis step then applies the decision guidance from Appendix A.

In [ ]:
def money(value):
    return round(float(value) + 1e-9, 2)

def policy_lookup(claim):
    refs = set(['POL-TIME-01', 'POL-APR-01', 'POL-APR-02', 'POL-APR-03'])
    for item in claim['items']:
        category = item['category']
        refs.update(POLICY['rules'].get(category, POLICY['rules']['ineligible']))
    return {'tool': 'policy_lookup', 'policy_refs': sorted(refs), 'context': 'Retrieved category eligibility, limits, receipt rules, approval tiers, and timeliness rules.'}

def receipt_completeness_check(claim):
    missing = []
    findings = []
    for item in claim['items']:
        required = item['category'] in {'airfare', 'lodging'} or item['amount'] > POLICY['receipt_threshold']
        if required and not item['receipt_attached']:
            missing.append(f"Itemized receipt: {item['category']} — {item['description']}")
            findings.append({'category': item['category'], 'status': 'MISSING_REQUIRED_RECEIPT'})
        else:
            findings.append({'category': item['category'], 'status': 'RECEIPT_OK_OR_NOT_REQUIRED'})
    return {'tool': 'receipt_completeness_check', 'missing_docs': missing, 'manual_review': bool(missing), 'findings': findings, 'policy_refs': ['POL-RCT-01', 'POL-RCT-02']}

def limit_checker(claim):
    line_results = []
    for item in claim['items']:
        category, amount = item['category'], money(item['amount'])
        approved, deducted, pending, refs, reason = amount, 0.0, 0.0, set(), 'Within policy.'
        if category in POLICY['ineligible_categories']:
            approved, deducted, refs, reason = 0.0, amount, {'POL-CAT-02'}, 'Ineligible personal or non-reimbursable category.'
        elif category == 'airfare':
            refs.update({'POL-CAT-01', 'POL-AIR-01'})
            if item.get('class', '').lower() != 'economy':
                approved, deducted, pending, reason = 0.0, 0.0, amount, 'Airfare class is a policy exception; pre-approval may exist.'
        elif category == 'lodging':
            cap = POLICY['lodging_nightly_cap'] * item.get('nights', 1)
            approved, deducted, refs = min(amount, cap), max(amount - cap, 0.0), {'POL-CAT-01', 'POL-PD-02'}
            reason = f'Lodging cap is ${money(cap):.2f} for {item.get("nights", 1)} night(s).' if deducted else 'Within nightly lodging cap.'
        elif category == 'meals':
            cap = POLICY['meal_daily_cap'] * item.get('days', 1)
            approved, deducted, refs = min(amount, cap), max(amount - cap, 0.0), {'POL-CAT-01', 'POL-PD-01'}
            reason = f'Meal cap is ${money(cap):.2f} for {item.get("days", 1)} day(s).' if deducted else 'Within daily meal cap.'
        elif category == 'ground_transport':
            cap = POLICY['ground_daily_cap'] * item.get('days', 1)
            approved, deducted, refs = min(amount, cap), max(amount - cap, 0.0), {'POL-CAT-01', 'POL-PD-03'}
            reason = f'Ground transport cap is ${money(cap):.2f} for {item.get("days", 1)} day(s).' if deducted else 'Within daily ground transport cap.'
        elif category in POLICY['eligible_categories']:
            refs.update({'POL-CAT-01'})
        else:
            approved, deducted, refs, reason = 0.0, amount, {'POL-CAT-02'}, 'Unknown category treated as not auto-reimbursable.'
        line_results.append({'category': category, 'description': item['description'], 'claimed': amount, 'approved_by_limit': money(approved), 'deducted_by_limit': money(deducted), 'pending_exception': money(pending), 'policy_refs': sorted(refs), 'reason': reason})
    return {'tool': 'limit_checker', 'line_results': line_results, 'approved_by_limit': money(sum(x['approved_by_limit'] for x in line_results)), 'deducted_by_limit': money(sum(x['deducted_by_limit'] for x in line_results)), 'pending_exception': money(sum(x['pending_exception'] for x in line_results)), 'policy_refs': sorted({r for x in line_results for r in x['policy_refs']}), 'manual_review': any(x['pending_exception'] > 0 for x in line_results)}

def approval_threshold_check(claim, limit_result):
    threshold_base = money(limit_result['approved_by_limit'] + limit_result['pending_exception'])
    if threshold_base <= 500:
        tier, manual = 'AUTO_APPROVE_TIER', False
    elif threshold_base <= 2000:
        tier, manual = 'MANAGER_TIER', False
    else:
        tier, manual = 'DIRECTOR_MANUAL_REVIEW_TIER', True
    return {'tool': 'approval_threshold_check', 'threshold_base': threshold_base, 'tier': tier, 'manual_review': manual, 'policy_refs': ['POL-APR-01', 'POL-APR-02', 'POL-APR-03']}

def timeliness_checker(claim):
    latest_expense = date.fromisoformat(claim['trip_end'])
    submitted = date.fromisoformat(claim['submitted'])
    age_days = (submitted - latest_expense).days
    late = age_days > POLICY['submission_window_days']
    return {'tool': 'timeliness_checker', 'days_after_latest_expense': age_days, 'late': late, 'manual_review': late, 'policy_refs': ['POL-TIME-01']}

def output_validator(result):
    if set(result) != REQUIRED_FIELDS:
        raise ValueError(f'Output fields must be exactly {sorted(REQUIRED_FIELDS)}')
    if result['decision'] not in ALLOWED_DECISIONS:
        raise ValueError('Invalid decision enum')
    if result['approved_amount'] < 0 or result['deducted_amount'] < 0:
        raise ValueError('Amounts cannot be negative')
    if not 0 <= result['confidence'] <= 1:
        raise ValueError('Confidence must be between 0 and 1')
    return result

def optional_llm_narration(evidence, fallback):
    # Kept optional so the assignment runs without credentials or network access.
    if not ENABLE_LLM_NARRATION or not os.getenv('OPENAI_API_KEY'):
        return fallback
    # The deterministic fallback remains the source of truth if an optional call fails.
    try:
        from urllib.request import Request, urlopen
        payload = {'model': os.getenv('OPENAI_MODEL', 'gpt-4o-mini'), 'input': 'Rewrite this policy evidence as one concise audit explanation. Do not change amounts or decision: ' + json.dumps(evidence)}
        request = Request('https://api.openai.com/v1/responses', data=json.dumps(payload).encode(), headers={'Content-Type': 'application/json', 'Authorization': 'Bearer ' + os.environ['OPENAI_API_KEY']})
        with urlopen(request, timeout=20) as response:
            body = json.loads(response.read().decode())
        return body.get('output_text', fallback) or fallback
    except Exception:
        return fallback

class TravelReimbursementAgent:
    def plan_tools(self, claim):
        tools = ['policy_lookup', 'receipt_completeness_check', 'limit_checker', 'approval_threshold_check', 'timeliness_checker']
        return tools

    def run(self, claim):
        tools_used = self.plan_tools(claim)
        policy_result = policy_lookup(claim)
        receipt_result = receipt_completeness_check(claim)
        limit_result = limit_checker(claim)
        threshold_result = approval_threshold_check(claim, limit_result)
        time_result = timeliness_checker(claim)
        evidence = {'policy': policy_result, 'receipts': receipt_result, 'limits': limit_result, 'threshold': threshold_result, 'timeliness': time_result}
        manual = receipt_result['manual_review'] or limit_result['manual_review'] or threshold_result['manual_review'] or time_result['manual_review']
        refs = set(policy_result['policy_refs']) | set(receipt_result['policy_refs']) | set(limit_result['policy_refs']) | set(threshold_result['policy_refs']) | set(time_result['policy_refs'])
        missing_docs = receipt_result['missing_docs']
        approved, deducted = limit_result['approved_by_limit'], limit_result['deducted_by_limit']
        if manual:
            decision, approved, deducted = 'MANUAL_REVIEW', 0.0, 0.0
            reasons = []
            if receipt_result['manual_review']: reasons.append('a required receipt is missing')
            if limit_result['manual_review']: reasons.append('airfare class is a policy exception')
            if threshold_result['manual_review']: reasons.append('the amount exceeds the agent authority')
            if time_result['manual_review']: reasons.append('the claim was submitted after the 30-day window')
            explanation = 'Manual review required because ' + ', '.join(reasons) + '. No amount is auto-approved or deducted while the unresolved evidence is reviewed.'
            confidence = 0.97
        elif approved == 0 and deducted > 0:
            decision, explanation, confidence = 'REJECT', 'All claimed items are ineligible under POL-CAT-02; the full claimed amount is deducted.', 0.99
        elif deducted > 0:
            decision, explanation, confidence = 'PARTIAL_APPROVE', f'Eligible items are reimbursed up to policy caps. ${deducted:.2f} is deducted for amounts above per-diem or category limits.', 0.99
        else:
            decision, explanation, confidence = 'APPROVE', f'All items are eligible, supported, timely, and within an approvable tier. ${approved:.2f} is approved.', 0.99
        explanation = optional_llm_narration(evidence, explanation)
        result = {'claim_id': claim['claim_id'], 'decision': decision, 'approved_amount': money(approved), 'deducted_amount': money(deducted), 'missing_docs': missing_docs, 'policy_refs': sorted(refs), 'confidence': confidence, 'explanation': explanation, 'tools_used': tools_used}
        return output_validator(result), evidence

agent = TravelReimbursementAgent()
results, audit_trail = [], {}
for claim in CLAIMS:
    result, evidence = agent.run(claim)
    results.append(result)
    audit_trail[claim['claim_id']] = evidence

assert len(results) == 5
results

## Sample outputs and audit trail

The compact table below demonstrates all four decision types across the provided claims. The audit trail keeps intermediate tool evidence available for review.

In [ ]:
for result in results:
    print(f"{result['claim_id']}: {result['decision']:<15} approved=${result['approved_amount']:>7.2f} deducted=${result['deducted_amount']:>7.2f}")

print('\nExample detailed result:')
print(json.dumps(results[0], indent=2))
print('\nAudit tools for CLM-004:')
print(json.dumps(audit_trail['CLM-004'], indent=2))

## Dashboard

This dashboard is derived from the actual structured results, not hard-coded expected outcomes.

In [ ]:
from collections import Counter

decision_counts = Counter(r['decision'] for r in results)
total_approved = money(sum(r['approved_amount'] for r in results))
total_deducted = money(sum(r['deducted_amount'] for r in results))
print('Decision breakdown:', dict(decision_counts))
print(f'Total approved: ${total_approved:,.2f} | Total deducted: ${total_deducted:,.2f}')

try:
    import matplotlib.pyplot as plt
    labels = list(decision_counts)
    values = [decision_counts[label] for label in labels]
    colors = {'APPROVE': '#2E7D32', 'PARTIAL_APPROVE': '#F9A825', 'REJECT': '#C62828', 'MANUAL_REVIEW': '#1565C0'}
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), gridspec_kw={'width_ratios': [1, 1.4]})
    axes[0].pie(values, labels=labels, autopct='%1.0f%%', startangle=90, colors=[colors[x] for x in labels], textprops={'fontsize': 9})
    axes[0].set_title('Decision breakdown')
    claim_labels = [r['claim_id'] for r in results]
    approved_values = [r['approved_amount'] for r in results]
    deducted_values = [r['deducted_amount'] for r in results]
    x = list(range(len(results)))
    axes[1].bar(x, approved_values, label='Approved', color='#2E7D32')
    axes[1].bar(x, deducted_values, bottom=approved_values, label='Deducted', color='#C62828')
    axes[1].set_xticks(x, claim_labels)
    axes[1].set_ylabel('USD')
    axes[1].set_title('Claim outcome amounts')
    axes[1].legend(frameon=False)
    fig.suptitle('Travel Reimbursement Approval Agent', fontsize=14, fontweight='bold')
    fig.tight_layout()
    fig.savefig('UI_SS_1.png', dpi=160, bbox_inches='tight')
    plt.show()
except ImportError:
    print('Matplotlib is not installed; the numeric dashboard above remains available.')

## Design Notes & Reasoning

- Policy rules are authoritative and encoded as inspectable tools. This reduces hallucinated limits and gives every result stable `POL-*` citations.
- The agent returns `MANUAL_REVIEW` for missing required receipts, business/first-class airfare, late submissions, and totals above $2,000. It intentionally reports approved and deducted amounts as zero while unresolved evidence is pending; this avoids promising reimbursement before a reviewer decides.
- `PARTIAL_APPROVE` is used only when the claim is otherwise clear and a numeric cap can be applied, as with CLM-003 lodging. Ineligible-only claims become `REJECT`.
- Confidence measures consistency of the rule-based conclusion, not the chance of payment. Manual-review cases have high confidence that escalation is required.
- The optional LLM hook is limited to explanation writing. A production version would add authenticated policy storage, receipt OCR, duplicate detection across historical claims, human decision capture, and monitoring for policy changes.

### Expected policy outcomes

CLM-001 is approved for $1,110.00; CLM-002 is rejected for $0.00 because spa and minibar are ineligible; CLM-003 is partially approved for $840.00 with a $100.00 lodging deduction; CLM-004 is routed to manual review for business airfare, missing lodging receipt, and high value; CLM-005 is routed to manual review because the meal is over $25 and has no receipt.

## Final structured output

The next and final code cell emits one JSON object per provided claim with exactly the required fields.

In [ ]:
final_results = [output_validator(deepcopy(result)) for result in results]
print(json.dumps(final_results, indent=2))